# LLM Fine-Tuning Workflow using Unsloth & GGUF

This notebook contains the skeleton for performing end-to-end dataset extraction, model fine-tuning with Unsloth, and exporting the final model to GGUF.

---  
### Workflow Steps:
* **Step 1:** Download the Model which you want to train/tune (Unsloth downloads 4-bit base models directly).
* **Step 2:** Run the base model using Hugging Face/Jan/LM Studio to verify capability.
* **Step 3:** Define your purpose of model training and identify the domain (Legal, Healthcare, Finance, etc.).
* **Step 4:** Download/organize training files (`.docx`/`.pdf`/`.txt`).
* **Step 5:** Use Unsloth & Helper Smart Models to:
  * **a.** Extract datasets from the files.
  * **b.** Train the model on the dataset.
  * **c.** Export the trained model to GGUF format.

## Step 1 to 4: Domain & Material Preparation
1. **Identify Domain & Goal:** Define what tone, style, or specific knowledge base you want the model to learn.
2. **Collect Source files:** Save all raw `.txt`, `.docx`, or `.pdf` files in a folder.
3. **Select Base Model:** Unsloth trains 16-bit or 4-bit base models. Common choices for consumer hardware/Colab:
   - `unsloth/llama-3-8b-Instruct-bnb-4bit` (General purpose Llama-3)
   - `unsloth/Phi-3-mini-4k-instruct` (Fast, lightweight 3.8B model)

## Step 5a: Extract Dataset from Documents (using a Smart Model)
Since Unsloth expects training data in a structured format (like `Instruction-Input-Output` or conversational format), we can use a helper smart model (e.g. Gemini, GPT-4) to parse our documents and generate structured Q&A training examples.

In [ ]:
# Skeleton: How to use a helper model API (e.g., Google Gemini or OpenAI)
# to process text files and extract structured training JSONs

# !pip install google-generativeai

import os
import json

# Setup API Key
# os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY_HERE"

def extract_qa_from_text(text_chunk):
    """
    Sends raw text to a smart model API to generate question-answer pairs.
    """
    prompt = f"""
    You are an AI dataset generator. Read the text below and extract 5 high-quality Question & Answer pairs.
    Format your response EXACTLY as a JSON list:
    [
      {{"instruction": "Question content?", "input": "", "output": "Answer content."}}
    ]

    Text:
    {text_chunk}
    """
    # Replace with actual API call:
    # model = genai.GenerativeModel('gemini-1.5-flash')
    # response = model.generate_content(prompt)
    # return json.loads(response.text)
    
    # Placeholder return
    return [
        {"instruction": "Example question based on document text?", "input": "", "output": "Example answer."}
    ]

# Example text processing
raw_document_text = "This is a document about Legal contracts. Section 1 states that..."
dataset = extract_qa_from_text(raw_document_text)

# Save to a json file for training
with open("extracted_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print("Dataset saved to extracted_dataset.json")

## Step 5b: Fine-tune the model with Unsloth
We will load Unsloth, configure LoRA parameters, import the dataset, and run the training process.

In [ ]:
# Install Unsloth and relevant PyTorch/xformers dependencies
# Note: Unsloth runs best on Linux with NVIDIA GPUs (like Google Colab T4)

# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft transformers accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports RoPE scaling automatically
dtype = None # None for auto-detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

# Load the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Can choose Mistral or Phi-3
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN" # If model is gated (like Llama-3)
)

In [ ]:
# Configure Parameter-Efficient Fine-Tuning (PEFT/LoRA)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank (suggested 8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized for 0
    bias = "none",    # Optimized for "none"
    use_gradient_checkpointing = "unsloth", # Saves memory
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# Load your generated Q&A dataset
# dataset = load_dataset("json", data_files="extracted_dataset.json", split="train")

# Define prompt template for Llama-3-Instruct or ChatML format
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant trained on domain-specific files.<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output)
        texts.append(text)
    return { "text" : texts }

# mapped_dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = None, # Swap with mapped_dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Small steps for demo, adjust for dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Run SFT Trainer
# trainer_stats = trainer.train()

## Step 5c: Export to GGUF format
Now we can save the model. Unsloth has a built-in fast GGUF exporter.

In [ ]:
# Save the model to 16-bit GGUF, 8-bit GGUF, or 4-bit GGUF
# Common quantization methods: "q4_k_m", "q8_0", "f16"

# 1. Save to 8-bit GGUF locally (e.g. inside Google Colab)
# model.save_pretrained_gguf("model_q8", tokenizer, quantization_method = "q8_0")

# 2. Save and upload directly to Hugging Face
# model.push_to_hub_gguf("your_username/model_q8", tokenizer, quantization_method = "q8_0", token = "HF_WRITE_TOKEN")